# Problema del Peluquero Dormido — Análisis Formal y Verificación Experimental

**Universidad del Valle — Departamento de Ingeniería de Sistemas**  
**Sistemas Operativos — Taller Final**  

---

## Frente 1 — Reproducción Determinista del Fallo (*Lost Wakeup*)

### 1. Objetivo
El objetivo de esta sección es reproducir e ilustrar de manera completamente determinista el fallo de concurrencia conocido como **Lost Wakeup** (Despertar Perdido) en la versión incorrecta del problema del peluquero dormido, donde no se emplea una sincronización atómica para la secuencia de verificación de estado y bloqueo (dormir).

## 2. Fundamento Teórico

El fenómeno del **Lost Wakeup** (OSTEP, Capítulo 30) ocurre cuando un hilo lector/receptor se prepara para bloquearse esperando una señal, pero justo antes de efectuar la llamada de bloqueo (p. ej., `wait()`), es interrumpido por el planificador (scheduler). Otro hilo (el emisor) se ejecuta, altera la condición de espera y envía la señal de despertar (`signal` o `set`). Como el primer hilo aún no está técnicamente en estado de espera, la señal se pierde. Cuando el primer hilo reanuda su ejecución, procede a bloquearse y queda suspendido indefinidamente porque la señal ya fue emitida y no volverá a ocurrir.

En el problema del peluquero dormido:
1. El barbero comprueba `waiting == 0` (no hay clientes).
2. Justo antes de irse a dormir (ejecutar `wait()`), llega un cliente.
3. El cliente ve que el barbero está "despierto" (pues el estado aún no es "durmiendo"), por lo que no le envía la señal de despertar (`set()`), limitándose a sentarse en la silla de espera.
4. El barbero reanuda su ejecución, asume que no hay clientes porque ya hizo la evaluación previa, y se duerme (`wait()`).
5. El barbero queda durmiendo indefinidamente (Lost Wakeup) y el cliente esperando en la silla para siempre.

In [ ]:
# ===========================================================================
# FRENTE 1 — Implementación  del Problema del Peluquero Dormido
# Propósito: Demostrar el Lost Wakeup de forma determinista
# ===========================================================================
# Se importan las librerias
import threading
import time
import random

# Parámetros de la barbería
NUM_CHAIRS = 3          # Número de sillas de espera
INJECT_DELAY = 0.15     # Retardo inyectado para simular la interrupción del scheduler

# Variables de estado compartidas sin protección adecuada de exclusión mutua
waiting = 0             # Clientes esperando
barber_sleeping = False # Estado del barbero
barber_event = threading.Event()  # Evento para despertar al barbero

# Registro de eventos
event_log = []
log_lock = threading.Lock()  # Exclusión mutua únicamente para el orden del log impreso

def log(actor, message):
    """Registra un evento con marca de tiempo precisa."""
    with log_lock:
        ts = time.perf_counter()
        entry = f"[T={ts:.4f}s] [{actor:>12}] {message}"
        event_log.append(entry)
        print(entry)

def barber_incorrect():
    global waiting, barber_sleeping
    
    log("BARBERO", "Inicio de turno. Verificando si hay clientes...")
    
    # Verificación del estado de la sala
    # Entre esta lectura de 'waiting' y el bloqueo físico en 'wait()'
    # ocurre la interrupción del scheduler (ventana de vulnerabilidad).
    if waiting == 0:
        log("BARBERO", "No hay clientes. Me preparo para dormir...")
        
        # INYECCIÓN DEL RETARDO:
        # Forzamos al scheduler a dar paso a otro hilo (el cliente) antes de bloquearnos.
        time.sleep(INJECT_DELAY)
        
        # Transición al estado durmiendo y bloqueo
        barber_sleeping = True
        log("BARBERO", "Entrando en estado durmiendo. Ejecutando wait()...")
        
        # Usamos un timeout de 2.0 segundos
        
        woke_up = barber_event.wait(timeout=2.0)
        
        if woke_up:
            barber_sleeping = False
            log("BARBERO", "Desperté. Atendiendo al cliente.")
        else:
            log("BARBERO", " FALLO: Nadie me despertó. Lost Wakeup confirmado.")
    else:
        log("BARBERO", f"Hay {waiting} cliente(s) esperando. Atendiendo...")

def client_incorrect(client_id, arrival_delay):
    global waiting, barber_sleeping
    
    # Simula el viaje del cliente a la barbería
    time.sleep(arrival_delay)
    log(f"CLIENTE-{client_id}", "Llegando a la barbería...")
    
    if waiting < NUM_CHAIRS:
        waiting += 1
        log(f"CLIENTE-{client_id}", f"Me senté en una silla. Esperando = {waiting}")
        
        # El cliente comprueba si el barbero duerme
        # PROBLEMA: Como el barbero está en su ventana de retardo, barber_sleeping es aún False.
        # El cliente asume erróneamente que el barbero está despierto y NO envía la señal.
        if barber_sleeping:
            log(f"CLIENTE-{client_id}", "El barbero está durmiendo. Enviando señal de despertar (set)... ")
            barber_event.set()
        else:
            log(f"CLIENTE-{client_id}", "⚠️ Barbero despierto. NO envío señal. [CAMINO DEL FALLO]")
    else:
        log(f"CLIENTE-{client_id}", "Sala llena. Me retiro.")

### 3. Ejecución Experimental del Fallo

In [ ]:
print("=== EXPERIMENTO: REPRODUCCIÓN DEL LOST WAKEUP ===\n")

# Limpieza e inicialización
event_log.clear()
barber_event.clear()
waiting = 0
barber_sleeping = False

# Hilos: El barbero inicia de inmediato (T=0). El cliente llega en T=0.05s, 
# cayendo exactamente en la ventana de vulnerabilidad del barbero (0s a 0.15s).
t_barber = threading.Thread(target=barber_incorrect)
t_client = threading.Thread(target=client_incorrect, args=(1, 0.05))

t_start = time.perf_counter()
t_barber.start()
t_client.start()

t_barber.join()
t_client.join()
t_duration = time.perf_counter() - t_start

print(f"\nDuración de la simulación: {t_duration:.4f} segundos")
if t_duration >= 2.0:
    print("\nRESULTADO: [Fallo Confirmado] El barbero quedó bloqueado permanentemente (se superó el timeout).")
else:
    print("\nRESULTADO: El barbero se despertó a tiempo.")

### 4. Análisis de la Intercalación Registrada
A continuación, podemos examinar la bitácora exacta de eventos para rastrear el entrelazamiento temporal (*interleaving*) que causó la pérdida de la señal de despertar:

In [ ]:
print("=== SECUENCIA DE INTERCALACIÓN REGISTRADA ===\n")
for entry in event_log:
    print(entry)

print("\nExplicación del entrelazamiento:")
print("1. El barbero inicia y verifica 'waiting == 0', lo cual es verdadero.")
print("2. El planificador suspende al barbero (simulado por INJECT_DELAY).")
print("3. El cliente llega, incrementa 'waiting' a 1, e inspecciona 'barber_sleeping'.")
print("4. Dado que el barbero aún no ha ejecutado 'wait()', 'barber_sleeping' es False.")
print("5. El cliente asume que el barbero lo atenderá y finaliza sin enviar la señal (set).")
print("6. El barbero reanuda, establece 'barber_sleeping = True' y se bloquea en 'wait()'.")
print("7. El barbero queda suspendido hasta que expira el timeout, demostrando el Lost Wakeup.")